In [0]:
%run ../../Configurations/Init_Scripts

Fraud Classification

In [0]:

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
input_path=ExternalLocation_raw+'/MasterData/FraudClassification'

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
output_path=ExternalLocationName_silver+'/DIMFraudClassification'

In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
# input_path = '/mnt/raw/MasterData/FraudClassification'
# output_path = '/mnt/silver/DIMFraudClassification'
df_FraudClassification = spark.read.csv(input_path,header=True)

df_Promotion = spark.read.table('promotion.dim_promotion').select('PromotionName','PromotionUUID')

df_FraudClassification = (df_FraudClassification.join(df_Promotion,'PromotionName','inner')
                            .withColumn('EffectiveDate', to_date(col('EffectiveDate'), 'MM/dd/yyyy'))
                            .withColumn('TerminationDate', to_date(col('TerminationDate'), 'MM/dd/yyyy'))
                            .drop('PromotionName')
                            )

In [0]:
df_FraudClassification_Insert = (df_FraudClassification.filter('Action = "New"')
                                                .withColumn('FraudClassificationUUID',expr('uuid()'))
                                                .withColumn('CreatedBy',lit('ADB_DIM_FraudClassification'))
                                                .withColumn('CreatedDate',current_timestamp())
                                                .withColumn('UpdatedBy',lit('ADB_DIM_FraudClassification'))
                                                .withColumn('UpdatedDate',current_timestamp()) 
                                 )  

In [0]:
df_FraudClassification_Target = (spark.read.load(output_path)
                                        .select('FraudClassificationUUID','PromotionUUID','RuleName','EffectiveDate')
                                        )

df_FraudClassification_Update = (df_FraudClassification.filter('Action = "Update"')
                                .join(df_FraudClassification_Target,['PromotionUUID','RuleName','EffectiveDate'],'left')
                                .withColumn('FraudClassificationUUID',coalesce(col('FraudClassificationUUID'),expr('uuid()')))
                                .withColumn('CreatedBy',lit('ADB_DIM_FraudClassification'))
                                .withColumn('CreatedDate',current_timestamp())
                                .withColumn('UpdatedBy',lit('ADB_DIM_FraudClassification'))
                                .withColumn('UpdatedDate',current_timestamp())
                                        )


In [0]:
df_FraudClassification_Upsert = df_FraudClassification_Update.unionByName(df_FraudClassification_Insert,allowMissingColumns=True)

In [0]:
display(df_FraudClassification_Upsert)

In [0]:
dl_Target = DeltaTable.forPath(spark, output_path)  
(dl_Target.alias("tgt")
  .merge(df_FraudClassification_Upsert.alias("src"), "src.FraudClassificationUUID = tgt.FraudClassificationUUID")
  .whenMatchedUpdate(
    set = {
        "RuleCategory": "src.RuleCategory",
        "RuleFrequency": "src.RuleFrequency",
        "RuleType": "src.RuleType",
        "RuleDetails": "src.RuleDetails",
        "BaseThresholdLimit": "src.BaseThresholdLimit",
        "HighPerformerThresholdLimit": "src.HighPerformerThresholdLimit",
        "tgt.DisplayFlag": "src.DisplayFlag",
        "tgt.TerminationDate": "src.TerminationDate",
        "tgt.UpdatedDate": "src.UpdatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy"
           }
  )
 .whenNotMatchedInsert(
     values={
         "FraudClassificationUUID": "src.FraudClassificationUUID",
         "PromotionUUID": "src.PromotionUUID",
         "RuleName": "src.RuleName",
         "RuleDetails": "src.RuleDetails",
         "DisplayFlag": "src.DisplayFlag",
         "BaseThresholdLimit": "src.BaseThresholdLimit",
         "HighPerformerThresholdLimit": "src.HighPerformerThresholdLimit",
         "RuleCategory": "src.RuleCategory",
         "RuleFrequency": "src.RuleFrequency",
         "RuleType": "src.RuleType",
         "EffectiveDate": "src.EffectiveDate",
         "TerminationDate": "src.TerminationDate",
         "CreatedBy": "src.CreatedBy",
         "CreatedDate": "src.CreatedDate",
         "UpdatedBy": "src.UpdatedBy",
         "UpdatedDate": "src.UpdatedDate"
     }
 )
.execute()
)

In [0]:
# display(spark.read.load(output_path))

In [0]:
# df_FraudClassification_Target = (spark.read.load(output_path)
#                                         # .filter('current_date() between EffectiveDate AND TerminationDate')
#                                         .select('FraudClassificationUUID','PromotionUUID','RuleName','EffectiveDate',col('BaseThresholdLimit').alias('BaseThresholdLimit_tgt'),col('HighPerformerThresholdLimit').alias('HighPerformerThresholdLimit_tgt'))
#                                         )

# df_FraudClassification_Final = (df_FraudClassification
#                                 .join(df_FraudClassification_Target,['PromotionUUID','RuleName','EffectiveDate'],'left')
#                                 .withColumn('UpdatedBy',lit('ADB_DIM_FraudClassification'))
#                                 .withColumn('UpdatedDate',current_timestamp())
#                                         )

# df_FraudClassification_Insert = (df_FraudClassification_Final.filter('FraudClassificationUUID is null')
#                                                 .withColumn('FraudClassificationUUID_New',expr('uuid()'))
#                                                 .withColumn('TerminationDate', to_date(lit('12/31/9999'), 'MM/dd/yyyy'))
#                                                 .withColumn('CreatedBy',lit('ADB_DIM_FraudClassification'))
#                                                 .withColumn('CreatedDate',current_timestamp())
#                                                 .withColumn('UpdatedBy',lit('ADB_DIM_FraudClassification'))
#                                                 .withColumn('UpdatedDate',current_timestamp()) 
#                                  )                           

# df_FraudClassification_Update = df_FraudClassification_Final.filter('FraudClassificationUUID is not null')

# df_FraudClassification_Upsert = df_FraudClassification_Update.unionByName(df_FraudClassification_Insert,allowMissingColumns=True).drop('BaseThresholdLimit_tgt','HighPerformerThresholdLimit_tgt','EffectiveDate_tgt','TerminationDate_tgt')